## Functions

In [4]:
from collections import defaultdict
from ast import literal_eval  # to convert string list ↦ python list
import random

# CORE FUNCTION
def sample_next_token(prob_dicts, sequence, n):
	prefix = ()
	# Get the current prefix (last n-1 words)
	if n > 1:
		prefix = tuple(sequence[-(n-1):])

	# In the probability dictionary, get possible next tokens for the given prefix
	next_tokens = prob_dicts[n].get(prefix, {})
	# If no next tokens found, back off to (n-1)-gram model
	if next_tokens == {}:
		return sample_next_token(prob_dicts, sequence, n-1)				# RECURSION: try with n-1-gram
	
	words = list(next_tokens.keys())
	probs = list(next_tokens.values())
	
	next_token = random.choices(words, weights=probs, k=1)[0]
	print(f"Current n: {n}, Current prefix: {prefix}, Next token: {next_token} ")
	return next_token

# TASK 1
def generative_ngram(n, mt, prob_json, nr_outputs=1):
	# Create a list for probability dictionaries
	prob_dicts = [{}]

	for k in range(1, n+1):
		prob_dict = {}
		for prefix_str, next_tokens in prob_json[k].items():
			prefix = literal_eval(prefix_str)
			prob_dict[prefix] = {}
			for token, prob in next_tokens.items():
				prob_dict[prefix][token] = prob
		prob_dicts.append(prob_dict)

	# Initialize output list
	output = [""] * nr_outputs

	for i in range(nr_outputs):
		print("NEW OUTPUT IN PROGRESS")
		if mt == "word":
			# Generate a sentence starting from <s> and ending at </s> based on the probability dictionary
			sequence = ["<s>"]
			
			# As long as the end symbol </s> is not reached, keep sampling the next word
			while sequence[-1] != "</s>":
				next_word = sample_next_token(prob_dicts, sequence, n)
				sequence.append(next_word)
			
			output[i] = " ".join(sequence)		# Convert list of generated words to a single string using space as separator

		else:
			# Generate a character sequence ending at ".", "!" or "?" based on the probability dictionary
			sequence = [""]
			# As long as none of the end symbols is reached, keep sampling the next character
			while sequence[-1] != "." and sequence[-1] != "!" and sequence[-1] != "?":
				next_char = sample_next_token(prob_dicts, sequence, n)
				sequence.append(next_char)
			
			output[i] = "".join(sequence)		# Convert list of generated characters to a single string

	return output


# TASK 2
def complete_sentence(sentence, n, prob_json):
	# Create a list for probability dictionaries
	prob_dicts = [{}]

	for k in range(1, n+1):
		prob_dict = {}
		for prefix_str, next_tokens in prob_json[k].items():
			prefix = literal_eval(prefix_str)
			prob_dict[prefix] = {}
			for token, prob in next_tokens.items():
				prob_dict[prefix][token] = prob
		prob_dicts.append(prob_dict)

	# Initialize output sequence with the given sentence start
	sequence = sentence.copy()
			
	# As long as the end symbol ".", "!" or "?" is not reached, keep sampling the next word
	while sequence[-1] != "." and sequence[-1] != "!" and sequence[-1] != "?":
		next_word = sample_next_token(prob_dicts, sequence, n)
		sequence.append(next_word)
		
	output = " ".join(sequence)		# Convert list of generated words to a single string using space as separator
	return output


# Gap Filler (not working currently)
def fill_gap(sentence, n, prob_json, nr_words):
    
    filled_sentence = sentence.copy()
    
    for i, token in enumerate(filled_sentence):
        if token != "<gap>":
            continue
        
        if n == 1: # if unigram, don't need prefix
            prefix = ()  
        else:
            # where the prefix should begin in the sentence, for an n-gram model, the prefix must contain n-1 previous words
            start = max(0, i-(n-1)) # index should not be negative
            prefix = tuple(filled_sentence[start:i])
        
        # look up probabilities for this prefix
        next_tokens = nprob_dict.get(prefix, {})
        
        # if no data available for this prefix, use unigram probabilities
        if not next_tokens:
            uni_tokens = uniprob_dict.get((), {})
            
            if uni_tokens:
                # sample randomly according to unigram probabilities
                words = list(uni_tokens.keys())
                probs = list(uni_tokens.values())
                best_word = random.choices(words, weights=probs, k=1)[0]
            else:
                best_word = "<unk>"
        
        else:
            # normal n-gram case: choose highest-prob word
            best_word = max(next_tokens, key=next_tokens.get)
        
        filled_sentence[i] = best_word # replaces the current <gap> with the best word (word with highest probability)
    
    return filled_sentence

In [5]:
import os
import json

N = {5}													# Fallback strategy: if a prefix does not exist in N-gram dictionary, back off to N-1-gram model
modeltype = {"word"}									# "word", "character"
categories = {"business", "entertainment", "politics", "sport", "tech", "all"}	# "business", "entertainment", "politics", "sport", "tech", "all"

sourcefolder = f"./../Probability_NGrams/"
targetfolder = f"./../Task1_GeneratedSequences/"

## Task 1: Generate new sequences

In [ ]:
import os
import json

N = {1, 2, 3, 4, 5}													# Fallback strategy: if a prefix does not exist in N-gram dictionary, back off to N-1-gram model
modeltype = {"word", "character"}										# "word", "character"
categories = {"business", "entertainment", "politics", "sport", "tech", "all"}	# "business", "entertainment", "politics", "sport", "tech", "all"

sourcefolder = f"./../Probability_NGrams/"
targetfolder = f"./../Task1_GeneratedSequences/"


NEW N-GRAM IN PROGRESS: character-1-gram with category all
NEW OUTPUT IN PROGRESS
NEW N-GRAM IN PROGRESS: character-2-gram with category all
NEW OUTPUT IN PROGRESS
NEW N-GRAM IN PROGRESS: character-3-gram with category all
NEW OUTPUT IN PROGRESS
NEW N-GRAM IN PROGRESS: character-4-gram with category all
NEW OUTPUT IN PROGRESS
NEW N-GRAM IN PROGRESS: character-5-gram with category all
NEW OUTPUT IN PROGRESS
NEW N-GRAM IN PROGRESS: word-1-gram with category all
NEW OUTPUT IN PROGRESS
NEW N-GRAM IN PROGRESS: word-2-gram with category all
NEW OUTPUT IN PROGRESS
NEW N-GRAM IN PROGRESS: word-3-gram with category all
NEW OUTPUT IN PROGRESS
NEW N-GRAM IN PROGRESS: word-4-gram with category all
NEW OUTPUT IN PROGRESS
NEW N-GRAM IN PROGRESS: word-5-gram with category all
NEW OUTPUT IN PROGRESS
NEW N-GRAM IN PROGRESS: character-1-gram with category sport
NEW OUTPUT IN PROGRESS
NEW N-GRAM IN PROGRESS: character-2-gram with category sport
NEW OUTPUT IN PROGRESS
NEW N-GRAM IN PROGRESS: character-3-g

In [ ]:

# Number of sequences to be generated
nr_outputs = 1

for category in categories:
	for mt in modeltype:
		for n in N:
			print(f"NEW N-GRAM IN PROGRESS: {mt}-{n}-gram with category {category}")
			
			# Initialize list of dictionaries
			prob_json = [""]  # Start with a dummy entry for 0-grams to align indices

			# Read in JSON probability file(s) and catch non-existing files
			for k in range(1, n+1):
				json_path = f"{sourcefolder}{mt}_{k}grams_{category}_probability.json"
				if not os.path.exists(json_path):
					print(f"Skipping (file not found): {json_path}")
					continue
			
				with open(json_path, "r", encoding="utf-8") as f:
					#print(f"Loading: {json_path}")
					dictionary = json.load(f)
					prob_json.append(dictionary)

			# Let n-gram model create sequences
			output = generative_ngram(n, mt, prob_json, nr_outputs)

			# Make the list of outputs into a single string with line breaks
			output = "\n\n".join(output)

			# Save outputs as TXT file
			output_path = f"{targetfolder}{category}_{mt}_{n}grams_generated.txt"
			with open(output_path, "w", encoding="utf-8") as f:
				f.write(str(output))

NEW N-GRAM IN PROGRESS: word-5-gram with category all
NEW OUTPUT IN PROGRESS
Current n: 2, Current prefix: ('<s>',), Next token: Owen 
Current n: 3, Current prefix: ('<s>', 'Owen'), Next token: determined 
Current n: 4, Current prefix: ('<s>', 'Owen', 'determined'), Next token: to 
Current n: 5, Current prefix: ('<s>', 'Owen', 'determined', 'to'), Next token: stay 
Current n: 5, Current prefix: ('Owen', 'determined', 'to', 'stay'), Next token: in 
Current n: 5, Current prefix: ('determined', 'to', 'stay', 'in'), Next token: Madrid 
Current n: 5, Current prefix: ('to', 'stay', 'in', 'Madrid'), Next token: England 
Current n: 5, Current prefix: ('stay', 'in', 'Madrid', 'England'), Next token: forward 
Current n: 5, Current prefix: ('in', 'Madrid', 'England', 'forward'), Next token: Michael 
Current n: 5, Current prefix: ('Madrid', 'England', 'forward', 'Michael'), Next token: Owen 
Current n: 5, Current prefix: ('England', 'forward', 'Michael', 'Owen'), Next token: has 
Current n: 5, Cur

: 

### Task 2: Sentence Completion / Gap-Filler

Fill "< gap >" tokens in a sentence using an n-gram model.  

- For each gap, the code first finds the prefix: if it’s a unigram model it uses an empty prefix, otherwise it takes the previous n−1 words as the prefix.

- It looks up which words can follow that prefix in the probability dictionary; if nothing is found, it falls back to the unigram probabilities.

- When the code falls back to unigram probabilities, it uses random sampling so that more frequent words are more likely—but not guaranteed—to be chosen, creating more natural and varied gap-fill results.

In [4]:
import json
import os
import json

N = {1, 2, 3, 4, 5}													# Fallback strategy: if a prefix does not exist in N-gram dictionary, back off to N-1-gram model
categories = {"business", "entertainment", "politics", "sport", "tech", "all"}	# "business", "entertainment", "politics", "sport", "tech", "all"
	
sourcefolder = f"./../Probability_NGrams/"
targetfolder = f"./../Task2_SequenceCompletion/"

nr_outputs = 5			# Number of completion suggestions to be generated
nr_words = 1			# Number of words to inserted for <gap>

gap_sentences = ( 
    ["The", "economy", "has", "<gap>", "this", "year", "."],
    ["The", "finance", "minister", "said", "the", "<gap>", ",", "which", "had", "<gap>", "intense", "<gap>", "from","several","EU", "members", ",", "would", "be", "<gap>", "before", "the", "final", "vote", "next", "month", "."],
    ["After", "months", "of", "<gap>", ",", "the", "agreement", "between", "the", "two", "companies", "was", "finally", "<gap>", ",", "paving", "the", "way", "for", "joint", "<gap>", "into", "renewable", "energy", "technologies", "."],
    ["Scientists", "reported", "that", "the", "newly", "<gap>", "exoplanet", "shows", "<gap>", "of", "an", "atmosphere", "rich", "in", "methane", ",", "a", "finding", "that", "could", "<gap>", "current", "models", "of", "planetary", "formation", "."]
)

sentences = ( 
    ["The", "economy", "has"],
    ["The", "finance", "minister", "said"],
    ["After", "months", "of"],
    ["Scientists", "reported", "that"]
)

for category in categories:
	for n in N:
		print(f"NEW N-GRAM IN PROGRESS: {n}-gram with category {category}")
	
		# Initialize list of dictionaries
		prob_json = [""]  # Start with a dummy entry for 0-grams to align indices

		# Read in JSON probability file(s) and catch non-existing files
		for k in range(1, n+1):
			json_path = f"{sourcefolder}word_{k}grams_{category}_probability.json"
			if not os.path.exists(json_path):
				print(f"Skipping (file not found): {json_path}")
				continue
			
			with open(json_path, "r", encoding="utf-8") as f:
				#print(f"Loading: {json_path}")
				dictionary = json.load(f)
				prob_json.append(dictionary)
		
		outputs = []
		for sentence in sentences:
			print("NEW SENTENCE IN PROGRESS")
			sentence_outputs = []
			for j in range(nr_outputs):
				print("NEW COMPLETION IN PROGRESS")
				#filled = fill_gaps(sentence, n, prob_json, nr_words)
				completed = complete_sentence(sentence, n, prob_json)
				sentence_outputs.append(completed)
					
			# Convert the list of suggestions for one sentence completion into a single string with line breaks
			outputs.append("\n".join(sentence_outputs))

		# Convert the list of single sentence solution sets into a single string with double line breaks
		output = "\n\n".join(outputs)

		# Save output as TXT file
		output_path = f"{targetfolder}{category}_word_{n}grams_completions.txt"
		with open(output_path, "w", encoding="utf-8") as f:
			f.write(str(output))

NEW N-GRAM IN PROGRESS: 1-gram with category all
NEW SENTENCE IN PROGRESS
NEW COMPLETION IN PROGRESS
NEW COMPLETION IN PROGRESS
NEW COMPLETION IN PROGRESS
NEW COMPLETION IN PROGRESS
NEW COMPLETION IN PROGRESS
NEW SENTENCE IN PROGRESS
NEW COMPLETION IN PROGRESS
NEW COMPLETION IN PROGRESS
NEW COMPLETION IN PROGRESS
NEW COMPLETION IN PROGRESS
NEW COMPLETION IN PROGRESS
NEW SENTENCE IN PROGRESS
NEW COMPLETION IN PROGRESS
NEW COMPLETION IN PROGRESS
NEW COMPLETION IN PROGRESS
NEW COMPLETION IN PROGRESS
NEW COMPLETION IN PROGRESS
NEW SENTENCE IN PROGRESS
NEW COMPLETION IN PROGRESS
NEW COMPLETION IN PROGRESS
NEW COMPLETION IN PROGRESS
NEW COMPLETION IN PROGRESS
NEW COMPLETION IN PROGRESS
NEW N-GRAM IN PROGRESS: 2-gram with category all
NEW SENTENCE IN PROGRESS
NEW COMPLETION IN PROGRESS
NEW COMPLETION IN PROGRESS
NEW COMPLETION IN PROGRESS
NEW COMPLETION IN PROGRESS
NEW COMPLETION IN PROGRESS
NEW SENTENCE IN PROGRESS
NEW COMPLETION IN PROGRESS
NEW COMPLETION IN PROGRESS
NEW COMPLETION IN PROGR